In [0]:
df = spark.read.table("shopkart.bronze.orders_dirty")

In [0]:
df2 = df.dropDuplicates(["order_id"])

In [0]:
from pyspark.sql import functions as F
df3 = df2.withColumn(
    "order_status", 
    F.lower(F.trim(F.coalesce(F.col("order_status"), F.lit("Unknown"))))
)


In [0]:
df3.show()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType,TimestampType


timestamp_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "_extracted_at"
]

for column_name in timestamp_columns:
    df4 = df3.withColumn(
        column_name,
        F.to_timestamp(F.col(column_name), "yyyy-MM-dd HH:mm:ss")
    )

In [0]:
df4.printSchema()

In [0]:
from pyspark.sql.types import IntegerType,TimestampType

df5 = df4.withColumn(
    "order_purchase_timestamp",
    df4["order_purchase_timestamp"].cast(TimestampType())
).withColumn(
    "order_approved_at",
    df4["order_approved_at"].cast(TimestampType())
).withColumn(
    "order_delivered_carrier_date",
    df4["order_delivered_carrier_date"].cast(TimestampType())
).withColumn(
    "order_delivered_customer_date",
    df4["order_delivered_customer_date"].cast(TimestampType())
).withColumn(
    "order_estimated_delivery_date",
    df4["order_estimated_delivery_date"].cast(TimestampType())
).withColumn(
    "_extracted_at",
    df4["_extracted_at"].cast(TimestampType())
)

df5.printSchema()

In [0]:
df6 = df5.withColumn("delivery_days", df5.order_delivered_customer_date - df5.order_purchase_timestamp)

In [0]:
from pyspark.sql import functions as F
df7 = df6.withColumn("is_valid_delivery", 
                             F.when(F.col("order_delivered_customer_date") < F.col("order_purchase_timestamp"),'True')
                                .otherwise("False"))

In [0]:
df7.show()

In [0]:
df7.write.format("delta").mode("overwrite").saveAsTable("shopkart.silver.orders")

In [0]:
# ---- SK-04: "before" profile of bronze.orders_dirty (run this again on silver.orders for "after") ----
from pyspark.sql import functions as F

df = spark.table("shopkart.silver.orders")
total = df.count()
profile = {
    "total rows":                total,
    "distinct order_id":         df.select("order_id").distinct().count(),
    "exact duplicate rows":      total - df.distinct().count(),
    "NULL order_status":         df.filter(F.col("order_status").isNull()).count(),
    "non-normalised status":     df.filter(F.col("order_status") != F.lower(F.trim("order_status"))).count(),
    "delivered before purchase": df.filter(F.to_timestamp("order_delivered_customer_date")
                                           < F.to_timestamp("order_purchase_timestamp")).count(),
}
profile["key duplicate rows (total - distinct ids)"] = total - profile["distinct order_id"]
for k, v in profile.items():
    print(f"{k:<45} {v:>7,}")

In [0]:
df8 = spark.table("shopkart.silver.orders")
df8.show()

In [0]:
df9 = df8.select("order_id","order_status","order_purchase_timestamp","customer_id")

In [0]:
df10 = spark.table("shopkart.bronze.customers")

In [0]:
df11 = df10.select("customer_city","customer_state","customer_id")

In [0]:
df11.show()

In [0]:
joined_df1 = df9.join(df11, on="customer_id", how="inner")

In [0]:
joined_df1.show()

In [0]:
df12 = spark.table("shopkart.bronze.product_category_name_translation")

In [0]:
df14 = spark.table("shopkart.bronze.products")

In [0]:
joined_df2 = df14.join(df12, on="product_category_name", how ="left")

In [0]:
df15 = joined_df2.select("product_category_name","product_category_name_english","product_id")

In [0]:
df15.show(5)

In [0]:
df16 = spark.table("shopkart.bronze.order_items")

In [0]:
joined_df3 =df16.join(df15, on="product_id" ,how="left")

In [0]:
df17= joined_df3.select("product_id","order_id","price","freight_value","product_category_name_english")

In [0]:
df17.show(5)

In [0]:
from pyspark.sql import functions as F
df18 = df17.withColumn("line_total", F.round(df17.price + df17.freight_value,2))

In [0]:
df18.show(5)

In [0]:
df19 = joined_df1.join(df18, on="order_id", how = "left")

In [0]:
df19.show(5)

In [0]:
df20 = df19.select("order_id","order_status","order_purchase_timestamp","customer_city","customer_state","product_category_name_english","price","freight_value","line_total")

In [0]:
 df20.write.format("delta").mode("overwrite").saveAsTable("shopkart.silver.order_lines")

In [0]:
print("order_items:", df8.count())
print("order_lines:", df20.count())

In [0]:
df20.show()

In [0]:

from pyspark.sql.functions import col
null_df = df20.filter(col("product_category_name_english").isNull())
null_df.show()

In [0]:
null_df.count()

# data validation rules

In [0]:
from pyspark.sql import functions as F


In [0]:
df_order_line = spark.table("shopkart.silver.order_lines")

In [0]:
df_order_line.count()

In [0]:
from pyspark.sql import functions as F
df_validated = df_order_line.withColumn(
        "validation_error", 
        F.array(
                F.when(
                     F.col("order_id").isNull(),
                     F.lit("order_id_is_null")
                ),
                F.when(
                       F.col("order_status") == "unknown",
                       F.lit("unknown_order_status")
                ),
                F.when(
                       F.col("customer_city").isNull(),
                       F.lit("Invalid_customer_city")
                ),
                F.when(
                       F.col("order_purchase_timestamp").isNull(),
                       F.lit("Invalid_order_purchase_timestamp")
                ),
                F.when(
                       F.col("customer_state").isNull(),
                       F.lit("Invalid_customer_state")
                ),
                 F.when(
                       F.col("product_category_name_english").isNull(),
                       F.lit("Invalid_product_category_name_english")
                ),
                 F.when(
                       F.col("price").isNull() | (F.col("price") <= 0),
                       F.lit("Invalid_price")
                ),
                 F.when(
                       F.col("freight_value").isNull() | (F.col("freight_value") <= 0),
                       F.lit("Invalid_price")
                ),
                   F.when(
                       F.col("line_total").isNull() | (F.col("price") <= 0),
                       F.lit("Invalid_line_total")
                ),
        )

)

In [0]:
df_validated.show(5)

In [0]:
df_validated = df_validated.withColumn(
    "validation_error",
        F.array_compact(F.col("validation_error"))
  )

In [0]:
df_validated.show(100)

In [0]:
df_validated = df_validated.withColumn(
    "is_valid",
    F.size(F.col("validation_error")) == 0
)

In [0]:
# Valid records
valid_df = df_validated.filter(
    F.col("is_valid") == True
)

# Invalid records
quarantine_df = df_validated.filter(
    F.col("is_valid") == False
)

In [0]:
valid_df.show()

In [0]:
valid_df.count()

In [0]:
quarantine_df = quarantine_df.withColumn("_quarantined_at", F.current_timestamp())


In [0]:
quarantine_df.show()

In [0]:
quarantine_df.count()

In [0]:
valid_df.write\
.format("delta")\
.mode("overwrite")\
.saveAsTable("shopkart.silver.order_lines")

In [0]:
quarantine_df.write \
    .format("delta") \
    .mode("overwrite")\
    .saveAsTable("shopkart.silver.order_lines_quarantine")

In [0]:
df_new_table1 = spark.read.table("shopkart.silver.order_lines")

In [0]:
df_new_table1.show()

In [0]:
df_new_table2 = spark.read.table("shopkart.silver.order_lines_quarantine")

In [0]:
df_new_table2.show()

In [0]:
print("valid:", valid_df.count())
print("quarantine:", quarantine_df.count())
print("input:", df_order_line.count())